In [92]:
from deck import full_euchre_deck
from dealer import Dealer
from numba import njit
from n_play_round import round1, next_round
from tree_search import find_best_opener, trick_played, n_trick_sim
import numpy as np


In [93]:
game = Dealer(deck=full_euchre_deck ,players=4)
# game.stack_deck(stack_cards=dealer_pick, player=4)
game.deal_cards()
hands5 = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
hands5

array([[[ 13,   0],
        [-10,   0],
        [  0,  -9],
        [ -9,   0],
        [  0, 135]],

       [[  0, -13],
        [  0, 120],
        [ 10,   0],
        [  0, 110],
        [ 14,   0]],

       [[-11,   0],
        [  9,   0],
        [-13,   0],
        [ 12,   0],
        [  0, -10]],

       [[ 11,   0],
        [  0, 140],
        [  0, 100],
        [  0, -12],
        [-12,   0]]])

In [94]:
best_opener = find_best_opener(hands=hands5, lead=0, tricks=5, previous_winners=np.array([]), sim_func=n_trick_sim)

[0.13719692 0.06576402 0.101983   0.07351351 0.11666667]


In [95]:
best_opener

np.int64(0)

In [96]:
# no njit for now, might not be possible
def find_best_response(
    hands: np.ndarray,
    lead: int,
    tricks: int,
    previous_winners: np.array,
    best_opener: int,
):

    responder = (lead + 1) % 2

    if tricks == 5:
        r2_leads, r2_hands = round1(
            hands_dealt=hands, chosen_card=best_opener, leader=lead
        )

        r1_response_res = np.zeros(r2_leads.shape, dtype=np.float64)

        for w in range(r2_leads.shape[0]):
            r3_leads, r3_hands, r2_score = next_round(
                current_hands=np.array([r2_hands[w]]),
                leads=np.array([r2_leads[w]]),
                game_round=2,
                game_score=np.array([r2_leads[w]]).reshape(-1, 1),
            )
            r4_leads, r4_hands, r3_score = next_round(
                current_hands=r3_hands,
                leads=r3_leads,
                game_round=3,
                game_score=r2_score,
            )
            r5_leads, r5_hands, r4_score = next_round(
                current_hands=r4_hands,
                leads=r4_leads,
                game_round=4,
                game_score=r3_score,
            )
            r6_leads, r6_hands, r5_score = next_round(
                current_hands=r5_hands,
                leads=r5_leads,
                game_round=5,
                game_score=r4_score,
            )

            results = r5_score.reshape(r5_score.shape[0], 5)

            meta_results = np.zeros(results.shape[0], dtype=np.int64)

            # if eval_position % 2 == 0:
            for i in range(len(results)):
                meta_results[i] = np.sum(results[i] % 2) >= 3

            r1_response_res[w] = np.mean(meta_results)

        if responder % 2 == 0:
            best_response = np.argmax(r1_response_res)
        else:
            best_response = np.argmin(r1_response_res)

        print(f'Trick Played: \n {trick_played(hands5, r2_hands[best_response])}')
        optimal_response = r2_hands[best_response]
        winner = r2_leads[best_response]

    else:
        print('need to develop response for 4 tricks and less')

    return optimal_response, winner


In [97]:
r2_optimal, winner_round1 = find_best_response(hands=hands5, lead=0, tricks=5, previous_winners=np.array([]), best_opener=best_opener)

Trick Played: 
 [[13  0]
 [10  0]
 [ 9  0]
 [11  0]]


In [108]:
winner_round1, r2_optimal

(np.int64(0),
 array([[[-10,   0],
         [  0,  -9],
         [ -9,   0],
         [  0, 135]],
 
        [[  0, -13],
         [  0, 120],
         [  0, 110],
         [ 14,   0]],
 
        [[-11,   0],
         [-13,   0],
         [ 12,   0],
         [  0, -10]],
 
        [[  0, 140],
         [  0, 100],
         [  0, -12],
         [-12,   0]]]))

In [109]:
r2_best_opener = find_best_opener(hands=r2_optimal, lead=winner_round1, tricks=4, previous_winners=np.array([winner_round1]), sim_func=n_trick_sim)

[0.03405573 0.06779661 0.04089219 0.25      ]


In [ ]:
r3_leads, r3_hands = round1(hands_dealt=r2_optimal, chosen_card=r2_best_opener, leader=r1_winner)

In [ ]:

# Here is where I will start tomorrow. Need to build out the best response case after round 1 is complete

responder=1

r2_response_res = np.zeros(r3_leads.shape, dtype=np.float64)

for w in range(r3_leads.shape[0]):

    r4_leads, r4_hands, r3_score = next_round(
        current_hands=np.array([r3_hands[w]]),
        leads=np.array([r3_leads[w]]),
        game_round=3,
        game_score=np.array([r3_leads[w]]).reshape(-1, 1),
    )
    
    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r3_score
    )
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    for i in range(len(results)):
        meta_results[i] = np.sum(results[i]%2 + (r1_winner%2))>=3

if responder % 2 == 0:
    best_response = np.argmin(r2_response_res)
else:
    best_response = np.argmax(r2_response_res)
    
print(best_response)

In [112]:
# trick_played(r2_optimal, r3_hands[best_response])

In [113]:
# r2_winner = r3_leads[best_response]

In [114]:
# r3_optimal =  r3_hands[best_response]

In [115]:
# r3_optimal

In [116]:
# r3_best_opener = find_best_opener(hands=r3_optimal, lead=r2_winner, tricks=3, previous_winners=np.array([r1_winner, r2_winner]), sim_func=n_trick_sim)

In [117]:
# r2_winner

In [118]:
# r4_leads, r4_hands = round1(hands_dealt=r3_optimal, chosen_card=r3_best_opener, leader=r2_winner)

In [119]:
# responder=1

# r3_response_res = np.zeros(r4_leads.shape, dtype=np.float64)

# for w in range(r4_leads.shape[0]):

#     r5_leads, r5_hands, r4_score = next_round(
#         current_hands=np.array([r4_hands[w]]),
#         leads=np.array([r4_leads[w]]),
#         game_round=4,
#         game_score=np.array([r4_leads[w]]).reshape(-1, 1),
#     )
    
#     r6_leads, r6_hands, r5_score = next_round(
#         current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
#     )
#     results = r5_score.reshape(r5_score.shape[0], 5)

#     meta_results = np.zeros(results.shape[0], dtype=np.int64)

#     for i in range(len(results)):
#         meta_results[i] = np.sum(results[i]%2 + (r1_winner%2)+(r2_winner%2))>=3

# if responder % 2 == 0:
#     best_response = np.argmin(r3_response_res)
# else:
#     best_response = np.argmax(r3_response_res)
    
# print(best_response)

In [120]:
# trick_played(r3_optimal, r4_hands[best_response])

In [121]:
# r3_winner = r4_leads[best_response]

In [122]:
# r3_winner

In [123]:
# (r1_winner % 2) + (r2_winner % 2) + (r3_winner % 2)